In [ ]:
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.types import Command
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_tavily import TavilySearch
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
import operator
import os
import json


load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_API_BASE_URL")
base_url_response = os.getenv("DASHSCOPE_API_URL_RESPONSE")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

checkpoint = InMemorySaver()
MAX_TOOL_CALL_ROUNDS = 3

class AgentState(MessagesState):
    final_report:str
    tool_call_rounds:int = 0


tavily_tool = TavilySearch(max_results=4)

@tool
def search_tool(query: str) -> str:
    """
    搜索工具，用于搜索互联网上的相关信息。
    """
    return tavily_tool.invoke(query)

tools = [search_tool]


llm = ChatOpenAI(
    model="qwen3.7-flash",
    api_key=api_key,
    base_url=base_url_response,
    temperature=0
).bind_tools(tools)

REACT_PROMPT = """
你是一名专业研究助手，严格遵循ReAct范式：Thought -> Action -> Observation。

任务：针对用户研究问题，判断当前已有的信息是否足够完成一份严谨的结构化报告。

【已有上下文】
用户研究问题：{user_query}
过往思考记录：{thought_chain}
已执行搜索：{search_queries}
已获取观测信息：{observations}

你的输出要求：
1. Thought：先写下你的思考，分析当前信息缺口，哪些事实缺失，是否需要再次搜索。
2. 如果信息不足，调用web_search工具，生成精准搜索query；最多允许3轮搜索，避免无限循环。
3. 如果信息已经足够充分，不要调用工具，设置 need_more_search=False，准备生成报告。
4. 禁止编造没有来源的事实，所有结论必须来自搜索观测结果。
"""

def react_node(state: AgentState):
    """
    反应节点，根据当前状态生成ReAct范式。
    """
    response = llm.invoke(state["messages"])
    return {"messages": response.content}

def tool_executor_node(state: AgentState):
    """
    工具执行器，根据当前状态执行工具调用。
    """
    last_message = state["messages"][-1]
    if not isinstance(last_msg, AIMessage):
        raise NodeInterrupt(f"预期最后一条消息为AIMessage，实际收到：{type(last_msg).__name__}")
    tool_message = []
    for call in last_msg.tool_calls:
        if call["name"] == "search_tool":
            tool_result = search_tool.invoke(call["args"])
            tm = ToolMessage(
                content = tool_result,
                tool_call_id=call["id"]
                )
            tool_message.append(tm)
    return {"messages": tool_message, "tool_call_rounds": state["tool_call_rounds"] + 1}
REPORT_INSTRUCTION = HumanMessage(content="""
你是专业研究分析师，基于全部对话与搜索结果输出一份Markdown研究报告。
# 研究报告
## 摘要
## 背景概述
## 关键发现
## 多维度分析
## 信息冲突与局限性
## 结论建议

禁止编造事实；存在观点冲突要列明不同立场；信息不足标注【信息有限】。
只输出报告正文。
""")

def generate_report_node(state: AgentState):
    resp = llm.invoke(state["messages"] + [REPORT_INSTRUCTION])
    return {"final_report": resp.content}

# ------------------------------
# 条件路由：ReAct循环控制器
# ------------------------------
def route_after_reasoning(state: AgentState):
    last_msg = state["messages"][-1]
    if state["tool_call_rounds"] >= MAX_TOOL_CALL_ROUNDS:
        return "generate_report"

    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        return "call_tool"
    return "generate_report"

def build_graph():
    graph = StateGraph(AgentState)

    graph.add_node("react", react_node)
    graph.add_node("call_tool", tool_executor_node)
    graph.add_node("generate_report", generate_report_node)

    graph.add_edge(START, "react")
    graph.add_conditional_edges(
        source= "react",
        path=route_after_reasoning,
        path_map={
            "call_tool": "call_tool",
            "generate_report": "generate_report"
        }
    )
    graph.add_edge("call_tool", "react")
    graph.add_edge("generate_report", END)
    return graph.compile(checkpointer=checkpoint)


if __name__ == "__main__":
    agent_graph = build_graph()
    config ={"configurable": {"thread_id": "1"}}

    response = agent_graph.invoke(
        input={"messages": [HumanMessage(content="目前最新的ai架构")],"final_report":"","tool_call_rounds":0},
        config=config
    )
    print(response["final_report"])

    response2 = agent_graph.invoke(
        input={"messages": [HumanMessage(content="重点对比LangGraph和AutoGen的开发体验差异")],"final_report":""},
        config=config
    )
    print(response2["final_report"])

    for i, msg in enumerate(response2["messages"]):
        print(f"\n[{i}] {type(msg).__name__}")
        if isinstance(msg, AIMessage) and msg.tool_calls:
            print(f"👉 tool_calls: {msg.tool_calls}")
        preview = msg.content[:250].replace("\n", " ")
        print(f"content: {preview} ...")


In [ ]:
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage,ToolMessage
from langchain_tavily import TavilySearch
from openai.types.responses import ResponseToolSearchOutputItem
from pydantic import BaseModel,Field
from datetime import datetime
from langgraph.errors import NodeInterrupt


from typing import TypedDict,Annotated,Any,List,Dict
from dotenv import load_dotenv

import os
import json
import sqlite3

load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url_response = os.getenv("DASHSCOPE_API_URL_RESPONSE")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
checkpointer = InMemorySaver()
MAX_TOOL_CALLS = 5


# 创建工具函数
# 搜索工具
TavilySearch_tool = TavilySearch(max_results="4")
# 数据库查询工具
class database_select_tool:
    def __init__(self, db_path:str = "gold_data.db"):
        self.db_path = db_path
    def select(self,table:str,columns:str="*",where:str ="",params:List[Any]= None,limit:int=100) -> Dict[str,Any]:
        """从数据库中查询数据。"""
        params = params or []
        sql_parts =[f"SELECT {columns} FROM {table}"]
        if where:
            sql_parts.append(f"WHERE {where}")
        sql_parts.append(f"LIMIT {limit}")
        sql = " ".join(sql_parts)
        try:
            conn = sqlite3.connect(self.db_path)
            conn.row_factory = sqlite3.Row
            cur = conn.cursor()
            cur.execute(sql, tuple(params))
            rows = [dict(r) for r in cur.fetchall()]
            conn.close()
            return {"success": True, "count": len(rows), "data": rows}
        except Exception as e:
            return {"success": False, "error": str(e), "data": []}

# 数据库查询工具参数
class DBQuerySchema(BaseModel):
    table: str = Field(description="要查询的数据库表名")
    columns: str = Field(description="要查询的列名，多个列用逗号分隔")
    where: str = Field(description="查询条件，可选")
    params: List[Any] = Field(description="查询条件的参数，可选")
    limit: int = Field(description="返回的最大行数，默认100")

db_client = database_select_tool("gold_data.db")

# 定义工具函数
@tool
def search_tool(query: str) -> str:
    """搜索工具，用于搜索互联网上的相关信息。"""
    return TavilySearch_tool.invoke(query)

@tool("query_db",args_schema=DBQuerySchema)
def query_db( table: str,
    columns: str,
    where: str,
    params: List[Any],
    limit: int = 50) -> Dict[str, Any]:
    """
    查询本地SQLite数据库 gold_data.db
    【重要】黄金持仓数据表名称固定为 gold_data，禁止使用其他表名！
    表字段：
        name:用户名；quantity:持有数量；unit:单位(g/Kg)
    示例：查询小红：table="gold_data", columns="name,quantity,unit", where="name=?", params=["小红"]
    注意：构造where条件时，值使用 ? 占位符，真实数值放入params数组，禁止直接拼接字符串！
    """
    return db_client.select(
        table=table,
        columns=columns,
        where=where,
        params=params,
        limit=limit
    )

@tool
def get_date_tool() -> str:
    """获取当前日期。"""
    day_date=datetime.now().strftime("%Y-%m-%d")
    return f"当前日期是：{day_date}"
    
tools = [search_tool,query_db,get_date_tool]

llm=ChatOpenAI(
    model="qwen3.7-flash",
    api_key=api_key,
    base_url=base_url_response,
    temperature=0,
    extra_body = {"enable_thinking": False}
).bind_tools(tools)

REACT_PROMPT = """
你是一个专业的黄金估值助手，严格遵循ReAct范式：Thought -> Action -> Observation。。
任务：针对用户提出的问题，并根据用户问题调用工具函数。

【已有上下文】
用户的问题：{user_query}
过往的思考记录{thought_chain}
工具函数：{tools}
已执行的搜索{search_queries}
已获取观测信息{observations}

你的输出要求：
1. Thought：先写下你的思考，分析当前信息缺口，哪些事实缺失，是否需要再次搜索。
2. 如果信息不足，调用web_search工具，生成精准搜索query；最多允许3轮搜索，避免无限循环。
3. 如果信息已经足够充分，不要调用工具，设置 need_more_search=False，准备生成报告回答问题。
4. 禁止编造没有来源的事实，所有结论必须来自搜索观测结果。

"""
# 工具映射
tool_map={
    "search_tool": search_tool,
    "query_db": query_db,
    "get_date_tool": get_date_tool
}

class AgentState(MessagesState):
    final_report: str 
    tool_calls_rounds: int


def react_node(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": response}

def tool_executor_node(state: AgentState):
    last_message = state["messages"][-1]

    if not isinstance(last_message, AIMessage):
        raise NodeInterrupt(f"预期AIMessage，实际：{type(last_message).__name__}")

    tool_message = []

    for call in last_message.tool_calls:
        print(f"工具调用: name={call['name']}, args={call['args']}")
        if call["name"] in tool_map:
            tool = tool_map[call["name"]]
            try:
                result = tool.invoke(call["args"])
            except Exception as e:
                result = f"工具调用失败：{str(e)}"
            tm = ToolMessage(
                content=result,
                tool_call_id=call["id"],
            )
        else:
            tm = ToolMessage(
                content=f"工具调用失败：{call['name']}",
                tool_call_id=call["id"],
            )
    tool_message.append(tm)
    return {"messages": tool_message,"tool_calls_rounds": state["tool_calls_rounds"] + 1}

REPORT_PROMPT = f"""
你是一个专业的贵金属投资专家，基于对话和搜索以及数据库结果生成一份报告。报告内容包括：
1. 问题描述
2. 搜索结果
3. 数据库查询结果
4. 分析结论
5. 建议操作

禁止编造事实；存在观点冲突要列明不同立场；信息不足标注【信息有限】。
只输出报告正文。
"""

def generate_report_node(state: AgentState):
    response = llm.invoke(state["messages"]+[REPORT_PROMPT])
    return {"final_report": response.content}

# 路由判断
def route(state: AgentState):
    last_message = state["messages"][-1]
    if state["tool_calls_rounds"] >= MAX_TOOL_CALLS:
        return "generate_report_node"

    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "call_tool"
    else:
        return "generate_report_node"


def build_graph():
    graph = StateGraph(AgentState)
    
    # 添加节点
    graph.add_node("react", react_node)
    graph.add_node("call_tool", tool_executor_node)
    graph.add_node("generate_report_node", generate_report_node)
    
    # 添加边
    graph.add_edge(START, "react")
    graph.add_conditional_edges(
        source = "react",
        path = route,
        path_map={
            "call_tool": "call_tool",
            "generate_report_node": "generate_report_node"
        }
    )
    graph.add_edge("call_tool", "react")
    graph.add_edge("generate_report_node", END)
    return graph.compile(checkpointer=checkpointer)

if __name__ == "__main__":
    agent_graph = build_graph()
    config = {"configurable":{"thread_id": "123"}}


    response = agent_graph.invoke(
        input={"messages": [HumanMessage(content="小红现在持有的黄金市值多少钱？")],"final_report": "","tool_calls_rounds":0},
        config=config
    )
    print(response["messages"][-1].content)



    response2 = agent_graph.invoke(
        input={"messages": [HumanMessage(content="你认为是否应该卖出还是增持呢？")],"final_report": ""},
        config=config
    )
    print(response2["messages"][-1].content)
    print(response2["final_report"])
